In [6]:
# First: Simulation of the probability of the group stage teams advancing to round of 32

import numpy as np
import pandas as pd
from collections import Counter

# 1. Official 48 Teams in the Group Stage with Elo ratings from https://eloratings.net/
elo_ratings = {
    "Argentina": 2148, "Spain": 2144, "France": 2123, "England": 2038,
    "Brazil": 2009, "Colombia": 2004, "Portugal": 1990, "Netherlands": 1980,
    "Norway": 1918, "Germany": 1916, "Switzerland": 1914, "Mexico": 1912,
    "Japan": 1910, "Croatia": 1905, "Ecuador": 1902, "Belgium": 1884,
    "Morocco": 1877, "Turkey": 1852, "Senegal": 1842, "Uruguay": 1841, 
    "Austria": 1836, "Ivory Coast": 1743, "Sweden": 1742, "Egypt": 1742, 
    "South Korea": 1723, "Paraguay": 1815, "Australia": 1800, "Algeria": 1785, 
    "United States": 1781, "Iran": 1764, "Canada": 1748, "Cabo Verde": 1622,
    "Bosnia and Herzegovina": 1622, "Ghana": 1575, "Panama": 1658, "South Africa": 1610, 
    "Czech Republic": 1710, "Qatar": 1530, "Scotland": 1690, "Haiti": 1490, 
    "Curaçao": 1450, "Tunisia": 1600, "New Zealand": 1510, "Saudi Arabia": 1620, 
    "Iraq": 1480, "Jordan": 1520, "DR Congo": 1640, "Uzbekistan": 1540
}

# 2. The 12 Official Groups
groups = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Switzerland", "Canada", "Bosnia and Herzegovina", "Qatar"],
    "C": ["Brazil", "Morocco", "Scotland", "Haiti"],
    "D": ["United States", "Australia", "Paraguay", "Turkey"],
    "E": ["Germany", "Ivory Coast", "Ecuador", "Curaçao"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cabo Verde", "Uruguay", "Saudi Arabia"],
    "I": ["France", "Norway", "Senegal", "Iraq"],
    "J": ["Argentina", "Austria", "Algeria", "Jordan"],
    "K": ["Colombia", "Portugal", "DR Congo", "Uzbekistan"],
    "L": ["England", "Croatia", "Ghana", "Panama"]
}

# The average goals per game in 2026
AVG_GOALS_PER_TEAM = 3.03 / 2

def play_match(team1, team2):
    elo_diff = elo_ratings[team1] - elo_ratings[team2]
    lambda1 = max(AVG_GOALS_PER_TEAM * (1 + elo_diff / 400), 0.4)
    lambda2 = max(AVG_GOALS_PER_TEAM * (1 - elo_diff / 400), 0.4)
    return np.random.poisson(lambda1), np.random.poisson(lambda2)

def simulate_group(teams):
    table = {t: {"pts": 0, "gd": 0, "gf": 0, "team": t} for t in teams}
    matchups = [(0,1), (2,3), (0,2), (1,3), (0,3), (1,2)]
    
    for idx1, idx2 in matchups:
        t1, t2 = teams[idx1], teams[idx2]
        g1, g2 = play_match(t1, t2)
        
        table[t1]["gf"] += g1
        table[t1]["gd"] += (g1 - g2)
        table[t2]["gf"] += g2
        table[t2]["gd"] += (g2 - g1)
        
        if g1 > g2:
            table[t1]["pts"] += 3
        elif g2 > g1:
            table[t2]["pts"] += 3
        else:
            table[t1]["pts"] += 1
            table[t2]["pts"] += 1
            
    return sorted(table.values(), key=lambda x: (x["pts"], x["gd"], x["gf"]), reverse=True)

def run_group_stage_only():
    advancing_teams = []
    all_thirds = []
    
    for group_letter, team_list in groups.items():
        detailed_stats = simulate_group(team_list)
        advancing_teams.append(detailed_stats[0]["team"])
        advancing_teams.append(detailed_stats[1]["team"])
        all_thirds.append(detailed_stats[2])
        
    best_thirds = sorted(all_thirds, key=lambda x: (x["pts"], x["gd"], x["gf"]), reverse=True)
    for i in range(8):
        advancing_teams.append(best_thirds[i]["team"])
        
    return advancing_teams

# Monte Carlo Loop:
NUM_SIMULATIONS = 10000
advancement_counter = Counter()

for _ in range(NUM_SIMULATIONS):
    round_of_32_qualifiers = run_group_stage_only()
    advancement_counter.update(round_of_32_qualifiers)

# Results
df_advancement = pd.DataFrame(advancement_counter.items(), columns=["Team", "Successful_Qualifications"])

for team in elo_ratings.keys():
    if team not in df_advancement["Team"].values:
        new_row = pd.DataFrame([{"Team": team, "Successful_Qualifications": 0}])
        df_advancement = pd.concat([df_advancement, new_row], ignore_index=True)

df_advancement = df_advancement.sort_values(by="Successful_Qualifications", ascending=False)
df_advancement["Chances of Making Round of 32 (%)"] = (df_advancement["Successful_Qualifications"] / NUM_SIMULATIONS) * 100

print("GROUP STAGE ADVANCEMENT PROBABILITIES:")
print(df_advancement.to_string(index=False, columns=["Team", "Chances of Making Round of 32 (%)"]))

GROUP STAGE ADVANCEMENT PROBABILITIES:
                  Team  Chances of Making Round of 32 (%)
             Argentina                              99.98
                 Spain                              99.90
                France                              99.84
               England                              99.59
                Brazil                              99.55
           Switzerland                              99.48
              Portugal                              99.37
              Colombia                              99.32
                Mexico                              98.74
           Netherlands                              98.62
               Belgium                              98.38
               Ecuador                              98.18
               Germany                              97.89
               Croatia                              96.98
               Morocco                              96.62
                 Japan           

In [7]:
# Now based on the actual round of 32 teams as of June 28 2026

import numpy as np
import pandas as pd

# Elo Ratings of all 32 qualified teams based on https://eloratings.net/
elo_ratings = {
    "Argentina": 2148, "Spain": 2144, "France": 2123, "England": 2038,
    "Brazil": 2009, "Colombia": 2004, "Portugal": 1990, "Netherlands": 1980,
    "Norway": 1918, "Germany": 1916, "Switzerland": 1914, "Mexico": 1912,
    "Japan": 1910, "Croatia": 1905, "Ecuador": 1902, "Belgium": 1884,
    "Morocco": 1877, "Austria": 1836, "Paraguay": 1815, "Australia": 1800,
    "Algeria": 1785, "USA": 1781, "Senegal": 1842, "Egypt": 1742,
    "Côte d'Ivoire": 1743, "Sweden": 1742, "Canada": 1748, "DR Congo": 1640,
    "Ghana": 1575, "Cabo Verde": 1622, "Bosnia and Herzegovina": 1622, "South Africa": 1610
}
# Historically, the goals scored per game decreases as rounds progress, so I will be using the historical average of goals scored per game.
AVG_GOALS_PER_TEAM = 2.69 / 2

def simulate_match(team1, team2):
    """Simulates a knockout match. Returns winner."""
    elo_diff = elo_ratings[team1] - elo_ratings[team2]
    lambda1 = max(AVG_GOALS_PER_TEAM * (1 + elo_diff / 400), 0.4)
    lambda2 = max(AVG_GOALS_PER_TEAM * (1 - elo_diff / 400), 0.4)
    
    g1 = np.random.poisson(lambda1)
    g2 = np.random.poisson(lambda2)
    
    if g1 > g2:
        return team1
    elif g2 > g1:
        return team2
    else:
        return np.random.choice([team1, team2]) # Shootout coin-flip

# Round-by-round survival frequencies
teams_list = list(elo_ratings.keys())
tracker = {t: {"R16": 0, "QF": 0, "SF": 0, "Final": 0, "Champ": 0} for t in teams_list}

NUM_SIMULATIONS = 10000

for _ in range(NUM_SIMULATIONS):
    # --- Round of 32 ---
    # Top Half of Bracket (Quadrant 1 & 2)
    m1 = simulate_match("South Africa", "Canada")
    m2 = simulate_match("Netherlands", "Morocco")
    m3 = simulate_match("Germany", "Paraguay")
    m4 = simulate_match("France", "Sweden")
    m5 = simulate_match("Belgium", "Senegal")
    m6 = simulate_match("USA", "Bosnia and Herzegovina")
    m7 = simulate_match("Spain", "Austria")
    m8 = simulate_match("Portugal", "Croatia")
    
    # Bottom Half of Bracket (Quadrant 3 & 4)
    m9 = simulate_match("Brazil", "Japan")
    m10 = simulate_match("Côte d'Ivoire", "Norway")
    m11 = simulate_match("Mexico", "Ecuador")
    m12 = simulate_match("England", "DR Congo")
    m13 = simulate_match("Switzerland", "Algeria")
    m14 = simulate_match("Colombia", "Ghana")
    m15 = simulate_match("Australia", "Egypt")
    m16 = simulate_match("Argentina", "Cabo Verde")

    r16_winners = [m1, m2, m3, m4, m5, m6, m7, m8, m9, m10, m11, m12, m13, m14, m15, m16]
    for t in r16_winners: tracker[t]["R16"] += 1

    # Round of 16
    q1 = simulate_match(m1, m2)   # South Africa/Canada vs Netherlands/Morocco
    q2 = simulate_match(m3, m4)   # Germany/Paraguay vs France/Sweden
    q3 = simulate_match(m5, m6)   # Belgium/Senegal vs USA/Bosnia
    q4 = simulate_match(m7, m8)   # Spain/Austria vs Portugal/Croatia
    
    q5 = simulate_match(m9, m10)  # Brazil/Japan vs Côte d'Ivoire/Norway
    q6 = simulate_match(m11, m12) # Mexico/Ecuador vs England/DR Congo
    q7 = simulate_match(m13, m14) # Switzerland/Algeria vs Colombia/Ghana
    q8 = simulate_match(m15, m16) # Australia/Egypt vs Argentina/Cabo Verde

    qf_winners = [q1, q2, q3, q4, q5, q6, q7, q8]
    for t in qf_winners: tracker[t]["QF"] += 1

    # Quarter-Finals
    s1 = simulate_match(q1, q2) 
    s2 = simulate_match(q3, q4) 
    s3 = simulate_match(q5, q6) 
    s4 = simulate_match(q7, q8) 

    sf_winners = [s1, s2, s3, s4]
    for t in sf_winners: tracker[t]["SF"] += 1

    # Semi-Final
    f1 = simulate_match(s1, s2) # Top Half Representative
    f2 = simulate_match(s3, s4) # Bottom Half Representative

    finalists = [f1, f2]
    for t in finalists: tracker[t]["Final"] += 1

    # Grand Final
    champion = simulate_match(f1, f2)
    tracker[champion]["Champ"] += 1
    
# Data
output_data = []
for t, metrics in tracker.items():
    output_data.append({
        "Team": t,
        "Make R16 (%)": (metrics["R16"] / NUM_SIMULATIONS) * 100,
        "Make QF (%)": (metrics["QF"] / NUM_SIMULATIONS) * 100,
        "Make SF (%)": (metrics["SF"] / NUM_SIMULATIONS) * 100,
        "Make Final (%)": (metrics["Final"] / NUM_SIMULATIONS) * 100,
        "Win Cup (%)": (metrics["Champ"] / NUM_SIMULATIONS) * 100
    })

df = pd.DataFrame(output_data).sort_values(by="Win Cup (%)", ascending=False)
print("BRACKET PREDICTIONS:")
print(df.to_string(index=False, formatters={
    "Make R16 (%)": "{:.1f}%".format, "Make QF (%)": "{:.1f}%".format,
    "Make SF (%)": "{:.1f}%".format, "Make Final (%)": "{:.1f}%".format,
    "Win Cup (%)": "{:.1f}%".format}))

BRACKET PREDICTIONS:
                  Team Make R16 (%) Make QF (%) Make SF (%) Make Final (%) Win Cup (%)
             Argentina        93.7%       84.0%       63.7%          46.5%       27.4%
                 Spain        88.6%       68.0%       59.7%          38.6%       23.8%
                France        91.0%       75.1%       58.5%          35.3%       21.0%
               England        90.9%       63.2%       39.4%          17.6%        7.5%
              Colombia        92.0%       63.5%       24.0%          12.6%        4.9%
                Brazil        65.0%       45.3%       24.1%           9.6%        3.8%
           Netherlands        65.8%       55.9%       21.3%           8.3%        3.0%
              Portugal        62.3%       20.1%       14.8%           6.0%        2.4%
                Norway        75.4%       30.7%       12.4%           3.6%        1.1%
               Germany        65.5%       16.7%        8.7%           3.0%        0.8%
           Switzerland

In [9]:
#This is just to see the probability of countries winning, without knowing the outcomes of the group stages (before we knew the countries for the actual round of 32).

import numpy as np
import pandas as pd
from collections import Counter

# 1. Official 48 Teams in the Group Stage with Elo ratings
elo_ratings = {
    "Argentina": 2148, "Spain": 2144, "France": 2123, "England": 2038,
    "Brazil": 2009, "Colombia": 2004, "Portugal": 1990, "Netherlands": 1980,
    "Norway": 1918, "Germany": 1916, "Switzerland": 1914, "Mexico": 1912,
    "Japan": 1910, "Croatia": 1905, "Ecuador": 1902, "Belgium": 1884,
    "Morocco": 1877, "Turkey": 1852, "Senegal": 1842, "Uruguay": 1841, 
    "Austria": 1836, "Ivory Coast": 1743, "Sweden": 1742, "Egypt": 1742, 
    "South Korea": 1723, "Paraguay": 1815, "Australia": 1800, "Algeria": 1785, 
    "United States": 1781, "Iran": 1764, "Canada": 1748, "Cabo Verde": 1622,
    "Bosnia and Herzegovina": 1622, "Ghana": 1575, "Panama": 1658, "South Africa": 1610, 
    "Czech Republic": 1710, "Qatar": 1530, "Scotland": 1690, "Haiti": 1490, 
    "Curaçao": 1450, "Tunisia": 1600, "New Zealand": 1510, "Saudi Arabia": 1620, 
    "Iraq": 1480, "Jordan": 1520, "DR Congo": 1640, "Uzbekistan": 1540
}

# 2. The 12 Official Groups
groups = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Switzerland", "Canada", "Bosnia and Herzegovina", "Qatar"],
    "C": ["Brazil", "Morocco", "Scotland", "Haiti"],
    "D": ["United States", "Australia", "Paraguay", "Turkey"],
    "E": ["Germany", "Ivory Coast", "Ecuador", "Curaçao"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cabo Verde", "Uruguay", "Saudi Arabia"],
    "I": ["France", "Norway", "Senegal", "Iraq"],
    "J": ["Argentina", "Austria", "Algeria", "Jordan"],
    "K": ["Colombia", "Portugal", "DR Congo", "Uzbekistan"],
    "L": ["England", "Croatia", "Ghana", "Panama"]
}

AVG_GOALS_PER_TEAM = 3.03 / 2

def play_match(team1, team2, knockout=False):
    elo_diff = elo_ratings[team1] - elo_ratings[team2]
    lambda1 = max(AVG_GOALS_PER_TEAM * (1 + elo_diff / 400), 0.4)
    lambda2 = max(AVG_GOALS_PER_TEAM * (1 - elo_diff / 400), 0.4)
    
    g1 = np.random.poisson(lambda1)
    g2 = np.random.poisson(lambda2)
    
    if g1 > g2:
        return team1
    elif g2 > g1:
        return team2
    else:
        if knockout:
            return np.random.choice([team1, team2]) # Penalty Shootout
        return "Draw", g1, g2

def simulate_group(teams):
    table = {t: {"pts": 0, "gd": 0, "gf": 0, "team": t} for t in teams}
    matchups = [(0,1), (2,3), (0,2), (1,3), (0,3), (1,2)]
    
    for idx1, idx2 in matchups:
        t1, t2 = teams[idx1], teams[idx2]
        res = play_match(t1, t2, knockout=False)
        
        if res == "Draw":
            # Unpack draw details handled by string fallback from earlier edits
            pass 
        
    # Standard group simulation tracking
    table = {t: {"pts": 0, "gd": 0, "gf": 0, "team": t} for t in teams}
    for idx1, idx2 in matchups:
        t1, t2 = teams[idx1], teams[idx2]
        elo_diff = elo_ratings[t1] - elo_ratings[t2]
        l1 = max(AVG_GOALS_PER_TEAM * (1 + elo_diff / 400), 0.4)
        l2 = max(AVG_GOALS_PER_TEAM * (1 - elo_diff / 400), 0.4)
        g1, g2 = np.random.poisson(l1), np.random.poisson(l2)
        
        table[t1]["gf"] += g1
        table[t1]["gd"] += (g1 - g2)
        table[t2]["gf"] += g2
        table[t2]["gd"] += (g2 - g1)
        if g1 > g2: table[t1]["pts"] += 3
        elif g2 > g1: table[t2]["pts"] += 3
        else:
            table[t1]["pts"] += 1
            table[t2]["pts"] += 1
            
    return sorted(table.values(), key=lambda x: (x["pts"], x["gd"], x["gf"]), reverse=True)

def run_full_tournament_predictions():
    # Phase 1: Group Stage
    group_winners = {}
    group_runners = {}
    all_thirds = []
    
    for letter, team_list in groups.items():
        standings = simulate_group(team_list)
        group_winners[letter] = standings[0]["team"]
        group_runners[letter] = standings[1]["team"]
        all_thirds.append(standings[2])
        
    best_thirds_list = sorted(all_thirds, key=lambda x: (x["pts"], x["gd"], x["gf"]), reverse=True)
    thirds = [t["team"] for t in best_thirds_list[:8]]
    # Fill remaining wildcard padding for safe array structures
    while len(thirds) < 8:
        thirds.append(best_thirds_list[4]["team"])

    # Phase 2: Create a balanced Round of 32 Bracket
    # Merging Group winners, runners up, and wildcards evenly into a 16-match slate
    r32_matches = [
        (group_winners["A"], thirds[0]), (group_runners["B"], group_runners["C"]),
        (group_winners["E"], thirds[1]), (group_winners["F"], group_runners["D"]),
        (group_winners["C"], thirds[2]), (group_runners["A"], group_runners["E"]),
        (group_winners["G"], thirds[3]), (group_winners["H"], group_runners["F"]),
        (group_winners["B"], thirds[4]), (group_runners["I"], group_runners["J"]),
        (group_winners["I"], thirds[5]), (group_winners["J"], group_runners["K"]),
        (group_winners["D"], thirds[6]), (group_runners["G"], group_runners["H"]),
        (group_winners["K"], thirds[7]), (group_winners["L"], group_runners["L"])
    ]
    
    # Run Bracket rounds
    r16_teams = [play_match(m[0], m[1], knockout=True) for m in r32_matches]
    
    qf_matches = [(r16_teams[i], r16_teams[i+1]) for i in range(0, len(r16_teams), 2)]
    qf_teams = [play_match(m[0], m[1], knockout=True) for m in qf_matches]
    
    sf_matches = [(qf_teams[i], qf_teams[i+1]) for i in range(0, len(qf_teams), 2)]
    sf_teams = [play_match(m[0], m[1], knockout=True) for m in sf_matches]
    
    final_teams = [play_match(sf_teams[0], sf_teams[1], knockout=True), play_match(sf_teams[2], sf_teams[3], knockout=True)]
    champion = play_match(final_teams[0], final_teams[1], knockout=True)
    
    return r16_teams, qf_teams, sf_teams, final_teams, champion

# --- Monte Carlo Execution ---
NUM_SIMULATIONS = 10000
track_r16, track_qf, track_sf, track_final, track_champ = Counter(), Counter(), Counter(), Counter(), Counter()

print(f"Processing {NUM_SIMULATIONS} bracket outcomes from your Group Stage foundation...")
for _ in range(NUM_SIMULATIONS):
    r16, qf, sf, fn, champ = run_full_tournament_predictions()
    track_r16.update(r16)
    track_qf.update(qf)
    track_sf.update(sf)
    track_final.update(fn)
    track_champ.update([champ])

# --- Collate Results Table ---
all_results = []
for team in elo_ratings.keys():
    all_results.append({
        "Team": team,
        "Make R16 (%)": (track_r16[team] / NUM_SIMULATIONS) * 100,
        "Make QF (%)": (track_qf[team] / NUM_SIMULATIONS) * 100,
        "Make SF (%)": (track_sf[team] / NUM_SIMULATIONS) * 100,
        "Make Final (%)": (track_final[team] / NUM_SIMULATIONS) * 100,
        "Win Cup (%)": (track_champ[team] / NUM_SIMULATIONS) * 100,
    })

df_final = pd.DataFrame(all_results).sort_values(by="Win Cup (%)", ascending=False)
print("PROJECTED REST-OF-WORLD-CUP OUTCOMES:")
print(df_final.to_string(index=False, formatters={
    "Make R16 (%)": "{:.1f}%".format, "Make QF (%)": "{:.1f}%".format,
    "Make SF (%)": "{:.1f}%".format, "Make Final (%)": "{:.1f}%".format,
    "Win Cup (%)": "{:.1f}%".format
}))

Processing 10000 bracket outcomes from your Group Stage foundation...
PROJECTED REST-OF-WORLD-CUP OUTCOMES:
                  Team Make R16 (%) Make QF (%) Make SF (%) Make Final (%) Win Cup (%)
                 Spain        84.3%       75.2%       59.3%          48.0%       29.7%
             Argentina        75.5%       48.9%       38.7%          28.9%       18.9%
                France        91.0%       52.7%       40.3%          28.8%       18.0%
               England        73.3%       44.2%       35.0%          14.5%        7.0%
                Brazil        82.5%       59.5%       27.9%          16.8%        6.8%
              Colombia        61.7%       30.5%       21.8%           8.7%        3.9%
           Netherlands        59.4%       39.2%       24.2%           9.3%        3.4%
              Portugal        56.8%       26.1%       17.6%           6.9%        2.9%
               Germany        70.3%       34.7%       14.9%           5.1%        1.4%
                Mexico